In [118]:
import math
import torch
from torch.nn import CrossEntropyLoss
import torch.nn as nn
from transformers import StoppingCriteria
from transformers import PreTrainedModel, PretrainedConfig
from transformers.modeling_outputs import CausalLMOutputWithCrossAttentions

from transformers import AutoModelForCausalLM, AutoTokenizer


In [119]:
import datasets
from datasets import load_dataset

In [120]:
bs = 4
l = 64
d = 128
mem_size = 1

sample_inputs = torch.rand((bs, l, d))
sample_memory = torch.rand((bs, mem_size, d))

In [121]:
class GatingLayer(nn.Module):
    def __init__(self, hidden_size) -> None:
        super().__init__()
        self.linear = nn.Linear(hidden_size, 1)
        self.act = nn.Sigmoid()
    
    def forward(self, hidden_states):
        return self.act(self.linear(hidden_states))

In [122]:
sample_inputs.shape, sample_memory.shape

(torch.Size([4, 64, 128]), torch.Size([4, 1, 128]))

In [123]:
# write to memory
write_gate = GatingLayer(d)
read_layer = GatingLayer(d)

# gating
gate_write_coefs = write_gate(sample_inputs)
gate_write_values = torch.einsum("bld, bld -> bld", sample_inputs, gate_write_coefs)

# aggregate by simple sum
bs, dim = sample_inputs.shape[0], sample_inputs.shape[2]
gate_write_values = gate_write_values.sum(dim=1).reshape(bs, 1, dim)

memory = sample_memory + gate_write_values

In [124]:
gate_write_values = torch.einsum("bld, bld -> bld", sample_inputs, gate_write_coefs)

In [125]:
gate_write_values2 = sample_inputs * gate_write_coefs


In [126]:
gate_write_values2 - gate_write_values

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 

In [117]:
memory.shape

torch.Size([4, 1, 128])

In [97]:
pool = nn.MaxPool1d(10)

In [98]:
x = torch.rand(2, 3, 4)

In [101]:
pool(x, dim=0)

TypeError: MaxPool1d.forward() got an unexpected keyword argument 'dim'

In [94]:
gate_write_coefs.shape, gate_write_values.shape

(torch.Size([4, 64, 1]), torch.Size([4, 64, 128]))

In [82]:
gate = GatingLayer(d)

In [ ]:

sigmoid = torch.nn.Sigmoid()
linear = torch.nn.Linear(d, 1)

# for each token we get a gate, how much of it do we want to keep
gate_weights = sigmoid(linear(sample_inputs))

# for each token multiply all dimensions to the gate value
gate_res = torch.einsum("bld,bld -> bld", sample_inputs, gate_weights)


In [27]:
sample_inputs.shape

torch.Size([4, 64, 128])

torch.Size([4, 64, 1])

In [72]:
sample_inputs *= 0
sample_inputs += 0.5

In [73]:
gate_weights *= 0
gate_weights += 2

In [74]:
gate_res = torch.einsum("bld,bld -> bld", sample_inputs, gate_weights)
gate_res.shape

torch.Size([4, 64, 128])

In [75]:
gate_res

tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 

In [41]:
ones = torch.ones(2, 3, 4)
torch.einsum('ijk -> i', ones), torch.einsum('ijk -> j', ones), torch.einsum('ijk -> k', ones),


(tensor([12., 12.]), tensor([8., 8., 8.]), tensor([6., 6., 6., 6.]))

$$ (A_{ijk})^{jk}  = \sum_{j} \sum_{k} A_{ijk} $$

In [51]:
ones = torch.ones(2, 3, 4)
twos = torch.ones(2, 3, 4)
torch.einsum("ibc,abc->ia", ones, twos)

tensor([[12., 12.],
        [12., 12.]])

$$ S_i^a = (A_{ijk}) (B^{ajk}) = \sum_j \sum_k A_{ijk} B_{ajk}$$

In [52]:
ones = torch.ones(2, 3, 4)
twos = torch.ones(2, 3, 4)
torch.einsum("ijk,abc->ia", ones, twos)

tensor([[144., 144.],
        [144., 144.]])

$$ S_i^a = (A_{ijk}) (B^{ajk}) = \sum_j \sum_k A_{ijk} B_{ajk}$$

In [46]:
res = torch.einsum("ijk,abs->ajb", ones, twos)
res.shape

torch.Size([2, 3, 3])

In [ ]:
mem_update = samaa


In [16]:
hiddens.shape

torch.Size([4, 64, 1])

In [ ]:
sigmoid

In [4]:
import torch.nn as nn
class GatingLayer(nn.Module):
    """Gating layer that gates the memory read and write attention weights."""

    def __init__(
        self,
        hidden_size: int,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.gating_layer = nn.Linear(hidden_size, hidden_size, activation=nn.Sigmoid())
    
    def forward(self, inputs, outputs) -> torch.Tensor:
        pass
        


In [3]:
model_name = "state-spaces/mamba2-1.3b"
tokenizer = AutoTokenizer.from_pretrained(model_name)

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [1]:
# ds = datasets.load_from_disk("/home/bulatov/rmt/test-time/compressing-associations/data/N2-K2V62-V62_1M")
# ds = datasets.load_from_disk("/home/bulatov/rmt/test-time/compressing-associations/data/N8-K2V62-V62_1M")

In [10]:
ds['train'][0]

{'context': '!yJ:8qUhQS83aq61Cp9UlOY5Kr9qTHSo6LgqOn5opxwaufYOUQxUUzPLg3L6dJwPlW!|',
 'query': '?!yJ:',
 'target': '8qUhQS83aq61Cp9UlOY5Kr9qTHSo6LgqOn5opxwaufYOUQxUUzPLg3L6dJwPlW!|'}

In [ ]:

base_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [5]:
# elif args.base_model == 'mamba':
from transformers import AutoConfig
config = AutoConfig.from_pretrained('state-spaces/mamba-130m-hf')
config.num_hidden_layers = 4
config.n_layer = 4
config.hidden_size = 128
config.d_model = 128
config.expand = 4
config.intermediate_size = config.expand * config.hidden_size
config.d_inner = config.expand * config.hidden_size
config.time_step_rank = math.ceil(config.hidden_size / 16)

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

In [6]:
model = AutoModelForCausalLM.from_config(config)

The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.


In [13]:
model.backbone.layers[0]

MambaBlock(
  (norm): MambaRMSNorm(128, eps=1e-05)
  (mixer): MambaMixer(
    (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
    (act): SiLUActivation()
    (in_proj): Linear(in_features=128, out_features=1024, bias=False)
    (x_proj): Linear(in_features=512, out_features=40, bias=False)
    (dt_proj): Linear(in_features=8, out_features=512, bias=True)
    (out_proj): Linear(in_features=512, out_features=128, bias=False)
  )
)

In [41]:
def add_memory_layers(base_layer, memory_read_layer, memory_write_layer, initial_memory_state):
    original_forward = base_layer.forward
    
    def new_forward(hidden_states, *args, memory_state=initial_memory_state, **kwargs):
        hidden_states = memory_read_layer(hidden_states, memory_state=memory_state)
        
        output = original_forward(hidden_states, *args, **kwargs)
        hidden_states = output[0] if isinstance(output, tuple) else output

        memory_state = memory_write_layer(hidden_states, memory_state)
        
        return (hidden_states,) + output[1:] if isinstance(output, tuple) else hidden_states
    
    base_layer.forward = new_forward
    base_layer.memory_state = initial_memory_state
    base_layer.memory_state.data = base_layer.memory_state


In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CrossAttention(nn.Module):
    def __init__(self, hidden_size, num_heads=8, dropout=0.1):
        super().__init__()
        assert hidden_size % num_heads == 0
        
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, from_states, to_states, attention_mask=None):
        batch_size, seq_len, _ = from_states.shape
        mem_len = to_states.shape[1]
        
        Q = self.q_proj(from_states)
        K = self.k_proj(to_states)
        V = self.v_proj(to_states)
        
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, mem_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, mem_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        
        if attention_mask is not None:
            attn_scores = attn_scores + attention_mask
        
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        
        attn_output = torch.matmul(attn_probs, V)
        attn_output = attn_output.transpose(1, 2).contiguous()
        
        attn_output = attn_output.view(batch_size, seq_len, self.hidden_size)
        output = self.o_proj(attn_output)
        
        return output

In [67]:
class MemoryAugmentedLayer(nn.Module):
    def __init__(self, base_layer, memory_read_layer, memory_write_layer, 
                 initial_memory_state):
        super().__init__()
        self.base_layer = base_layer
        self.memory_read = memory_read_layer
        self.memory_write = memory_write_layer
        
        self.register_buffer('memory_state', initial_memory_state)
    
    def forward(self, hidden_states, *args, **kwargs):
        # Read from memory
        print(f"[debug] reading from memory")
        hidden_states = self.memory_read(hidden_states, self.memory_state)
        
        # Base layer forward
        output = self.base_layer(hidden_states, *args, **kwargs)
        hidden_states = output[0] if isinstance(output, tuple) else output
        
        # Write to memory
        print(f"[debug] writing to memory")
        self.memory_state = self.memory_write(self.memory_state, hidden_states)
        
        return (hidden_states,) + output[1:] if isinstance(output, tuple) else hidden_states
    
    def reset_memory(self, new_state=None):
        """Helper to reset memory between sequences"""
        if new_state is not None:
            self.memory_state = new_state
        else:
            self.memory_state.zero_()

In [68]:
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [69]:
# Usage:
hidden_size = model.config.hidden_size
n_heads = 8
num_mem_tokens = 10

for i, layer in enumerate(model.model.layers):
    memory_read_layer = CrossAttention(hidden_size, n_heads).to(dtype=model.dtype, device=model.device)
    memory_write_layer = CrossAttention(hidden_size, n_heads).to(dtype=model.dtype, device=model.device)
    memory_state = torch.randn(1, num_mem_tokens, hidden_size, dtype=model.dtype, device=model.device) / math.sqrt(hidden_size)
    wrapped_layer = MemoryAugmentedLayer(
        layer, 
        memory_read_layer, 
        memory_write_layer,
        memory_state
    )
    model.model.layers[i] = wrapped_layer

In [70]:
# test model forward
test_input = "My name is"
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
test_input_ids = tokenizer(test_input, return_tensors="pt").input_ids
test_input_ids
test_input_ids.shape

model.eval()
with torch.no_grad():
    out = model(test_input_ids)
out
out.logits

[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory
[debug] writing to memory
[debug] reading from memory


tensor([[[-1.9688, 21.8750, 22.0000,  ..., 33.2500, 17.1250, 15.1875],
         [-2.0312, 22.0000, 22.1250,  ..., 33.2500, 17.1250, 15.2500],
         [-1.9688, 22.0000, 22.1250,  ..., 33.2500, 17.1250, 15.2500]]],
       dtype=torch.bfloat16)

In [71]:
print(tokenizer.decode(out.logits.argmax(dim=-1)[0]))

 affili affili affili


In [ ]:
print(tokenizer.decode(out.logits.argmax(dim=-1)[0]))

 own is John


In [ ]:


class RMTConfig(PretrainedConfig):
    model_type = "rmt"

    def __init__(self,
                 base_model_name="HuggingFaceTB/SmolLM2-135M",
                 base_model_config=None,
                 from_pretrained=None,
                 num_mem_tokens=16,
                 max_n_segments=10,
                 think_token_id=None,
                 answer_token_id=None,
                 bos_token_id=None,
                 eos_token_id=None,
                 **kwargs):
        super().__init__(**kwargs)
        self.base_model_name = base_model_name
        self.base_model_config = base_model_config
        self.from_pretrained = from_pretrained
        self.num_mem_tokens = num_mem_tokens
        self.max_n_segments = max_n_segments
        self.think_token_id = think_token_id
        self.answer_token_id = answer_token_id
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.memory_cell_cls = "MemoryCell"
        self.recurrent_wrapper_cls = "RecurrentWrapperNoSegmentationGenerate"

    def get(self, attr: str, default=None):
        if hasattr(self, attr):
            return getattr(self, attr)
        else:
            return default


class RMCAForReasoning(PreTrainedModel):
    config_class = RMTConfig

    def __init__(self, config: RMTConfig, **kwargs):
        super().__init__(config, **kwargs)
        from transformers import AutoConfig, AutoModelForCausalLM
        if config.from_pretrained:
            base_model = AutoModelForCausalLM.from_pretrained(config.from_pretrained)
        else:
            if config.base_model_config is None:
                base_config = AutoConfig.from_pretrained(config.base_model_name)
            else:
                base_config = config.base_model_config
            base_model = AutoModelForCausalLM.from_config(base_config)

        self.rmt_config = config
        memory_cell = RMCACell(base_model, num_mem_tokens=config.num_mem_tokens)
        self.rmt = RMCAWrapperNoSegmentation(
            memory_cell,
            max_n_segments=config.max_n_segments,
            think_token_id=config.think_token_id,
            answer_token_id=config.answer_token_id,
            bos_token_id=config.bos_token_id,
            eos_token_id=config.eos_token_id
        )

    def forward(self, labels=None, *args, **kwargs):
        return self.rmt(labels=labels, *args, **kwargs)

    def generate(self, *args, **kwargs):
        return self.rmt.generate(*args, **kwargs)

    def load_state_dict(self, state_dict, strict=True, assign=False):
        try:
            return super().load_state_dict(state_dict, strict, assign)
        except RuntimeError:
            print("Failed to load state, retrying with RMT loader.")
            self.rmt.load_state_dict(state_dict, strict=True, assign=assign)
            print("Success!")

    @classmethod
    def from_pretrained(cls, pretrained_model_name_or_path, config=None, *args, **kwargs):
        from transformers.utils.hub import cached_file, HfHubHTTPError
        import torch

        if config is None:
            config = RMTConfig.from_pretrained(pretrained_model_name_or_path, **kwargs)

        model = cls(config)

        state_dict = None
        try:
            weights_path = cached_file(pretrained_model_name_or_path, "model.safetensors", **kwargs)
            from safetensors.torch import load_file
            state_dict = load_file(weights_path, device="cpu")
        except (OSError, HfHubHTTPError):
            try:
                weights_path = cached_file(pretrained_model_name_or_path, "pytorch_model.bin", **kwargs)
                state_dict = torch.load(weights_path, map_location="cpu")
            except (OSError, HfHubHTTPError):
                print(f"Warning: Could not find weights for {pretrained_model_name_or_path}. "
                      f"The model is initialized randomly.")

        if state_dict is not None:
            model.load_state_dict(state_dict, strict=False)

        return model


class RMCACell(torch.nn.Module):
    def __init__(self, base_model, num_mem_tokens):
        super().__init__()
        self.model = base_model
        self.create_memory(num_mem_tokens)

    def create_memory(self, num_mem_tokens):
        self.num_mem_tokens = num_mem_tokens
        embeddings = self.model.get_input_embeddings()
        memory_dim = getattr(self.model.config, 'n_embd', self.model.config.hidden_size)
        memory_weights = torch.randn((num_mem_tokens, memory_dim)) * embeddings.weight.data.std()
        # self.register_parameter('memory', torch.nn.Parameter(memory_weights, requires_grad=True))

        # self.read_memory_position = range(num_mem_tokens)
        # self.write_memory_position = range(-num_mem_tokens, 0)

    def set_memory(self, input_shape):
        memory = self.memory.repeat(input_shape[0], 1, 1)
        return memory

    def forward(self, input_ids, memory_state=None, **kwargs):
        if memory_state is None:
            memory_state = self.set_memory(input_ids.shape)

        seg_kwargs = self.process_input(input_ids, memory_state, write_mem=True, **kwargs)
        out = self.model(**seg_kwargs)
        out, new_memory_state = self.process_output(out, **kwargs)

        return out, new_memory_state

    def generate(self, input_ids, memory_state, attention_mask=None, **generate_kwargs):
        if memory_state is None:
            memory_state = self.set_memory(input_ids.shape)

        seg_kwargs = self.process_input(input_ids, memory_state, attention_mask=attention_mask, write_mem=False)
        out = self.model.generate(inputs_embeds=seg_kwargs['inputs_embeds'],
                                  attention_mask=seg_kwargs['attention_mask'],
                                  **generate_kwargs)
        return out

    def process_input(self, input_ids, memory_state, write_mem, **kwargs):
        seg_kwargs = dict(**kwargs)

        inputs_embeds = kwargs.get('inputs_embeds')
        if inputs_embeds is None:
            inputs_embeds = self.model.get_input_embeddings()(input_ids)

        if self.num_mem_tokens > 0:
            if write_mem:
                inputs_embeds = torch.cat([memory_state, inputs_embeds, memory_state], dim=1)
            else:
                inputs_embeds = torch.cat([memory_state, inputs_embeds], dim=1)

        seg_kwargs['input_ids'] = None
        seg_kwargs['inputs_embeds'] = inputs_embeds
        if kwargs.get('attention_mask') is not None:
            seg_kwargs['attention_mask'] = self.pad_attention_mask(kwargs['attention_mask'], inputs_embeds.shape)
        seg_kwargs['output_hidden_states'] = True
        return seg_kwargs

    def pad_attention_mask(self, attention_mask, shape):
        if self.num_mem_tokens in {0, None}:
            return attention_mask
        else:
            mask = torch.ones(*shape[:2], dtype=torch.int64).to(attention_mask.device)
            mask[:, self.num_mem_tokens: self.num_mem_tokens + attention_mask.shape[1]] = attention_mask
            return mask

    def process_output(self, model_outputs, **kwargs):
        if self.num_mem_tokens not in {0, None}:
            out = CausalLMOutputWithCrossAttentions()
            memory_state = model_outputs.hidden_states[-1][:, -self.num_mem_tokens:]
            out['logits'] = model_outputs.logits[:, self.num_mem_tokens:-self.num_mem_tokens]

            if kwargs.get('output_hidden_states'):
                out['hidden_states'] = [lh[:, self.num_mem_tokens:-self.num_mem_tokens]
                                        for lh in model_outputs.hidden_states]
            if kwargs.get('output_attentions'):
                out['attentions'] = model_outputs['attentions']
        else:
            memory_state = None
            out = model_outputs

        return out, memory_state


class RMCAWrapperNoSegmentation(torch.nn.Module):
    def __init__(self, memory_cell, **rmt_kwargs):
        super().__init__()
        self.memory_cell = memory_cell
        self.rmt_config = rmt_kwargs

    def forward(self, segments, labels, output_attentions=None, output_hidden_states=None, *args, **kwargs):
        memory_state = None

        cell_outputs = []
        for seg_num, segment in enumerate(segments):
            cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
                                                      attention_mask=segment['attention_mask'],
                                                      memory_state=memory_state, output_hidden_states=True)
            cell_outputs.append(cell_out)
            self.manage_gradients(memory_state, seg_num)

        out = self.process_outputs(cell_outputs, segments,
                                   output_attentions=output_attentions,
                                   output_hidden_states=output_hidden_states)
        return out

    def generate(self, segments, **kwargs):
        memory_state = None

        for seg_num, segment in enumerate(segments):
            cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
                                                      attention_mask=segment['attention_mask'],
                                                      memory_state=memory_state, output_hidden_states=True)

        generated_segments = []
        for seg_num in range(len(segments), self.rmt_config.get("max_n_segments", 32)):
            output_ids, memory_state = self.generate_segment(memory_state=memory_state, **kwargs)
            generated_segments.append(output_ids)

            if self.all_done(generated_segments):
                break

        return generated_segments

    def generate_segment(self, memory_state, **kwargs):
        input_ids = self.get_bos_tensor(memory_state)
        attention_mask = torch.ones_like(input_ids).bool()

        generated = self.memory_cell.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            memory_state=memory_state,
            stopping_criteria=self.make_custom_stopping_criteria(),
            **kwargs
        )

        # Update memory state from generation
        fwd_inputs = torch.cat((input_ids, generated), dim=1)[:, :-1]
        _, memory_state = self.memory_cell(input_ids=fwd_inputs, memory_state=memory_state)

        return generated, memory_state

    def get_bos_tensor(self, memory_state):
        bos = self.rmt_config["bos_token_id"]
        bos_tensor = torch.tensor([bos] * memory_state.shape[0]).reshape(-1, 1)
        return bos_tensor.to(memory_state.device)

    def all_done(self, generated_segments):
        eos = self.rmt_config['eos_token_id']
        bs = generated_segments[0].shape[0]
        have_eos = [any([eos in seg[i] for seg in generated_segments]) for i in range(bs)]
        all_done = all(have_eos)
        return all_done

    def make_custom_stopping_criteria(self):
        return [StopOnSpecialTokenCriteria([self.rmt_config['think_token_id'], self.rmt_config['answer_token_id']])]

    def split_tensor(self, tensor):
        align = self.rmt_config.get('segment_alignment')
        segment_size = self.rmt_config.get('segment_size')
        if align in {'left', None}:
            split_inds = list(range(0, tensor.shape[1], segment_size)) + [tensor.shape[1]]
            segments = [tensor[:, start:end] for (start, end) in zip(split_inds, split_inds[1:])]
        elif align in {'right', None}:
            split_inds = (list(range(tensor.shape[1], 0, -segment_size)) + [0])[::-1]
            segments = [tensor[:, start:end] for (start, end) in zip(split_inds, split_inds[1:])]
        elif align == 'center':
            n_seg = math.ceil(tensor.shape[1] / segment_size)
            segments = torch.chunk(tensor, n_seg, dim=1)
        else:
            raise NotImplementedError
        return segments

    def process_outputs(self, cell_outputs, **kwargs):
        out = CausalLMOutputWithCrossAttentions()
        full_logits = torch.cat([o.logits for o in cell_outputs], dim=1)
        full_hidden_states = tuple([torch.cat(layer_hs, dim=1)
                                    for layer_hs in zip(*[o.hidden_states for o in cell_outputs])])

        labels = kwargs.get('labels')
        if labels is not None:
            shift_labels = labels[..., 1:].contiguous()
            shift_logits = full_logits[..., :-1, :].contiguous()
            flat_labels = shift_labels.view(-1)
            flat_logits = shift_logits.view(-1, shift_logits.size(-1))

            loss_fct = CrossEntropyLoss()
            labels_mask = kwargs.get('labels_mask')
            if labels_mask is not None:
                shift_mask = labels_mask[..., :-1].contiguous()

                flat_labels = flat_labels[shift_mask.view(-1)]
                flat_logits = flat_logits[shift_mask.view(-1)]

            out['loss'] = loss_fct(flat_logits, flat_labels)
        else:
            out['loss'] = 0

        out['logits'] = full_logits
        segment_keys = ['loss', 'logits']
        if kwargs.get('output_attentions'):
            segment_keys.append('attentions')
        if kwargs.get('output_hidden_states'):
            segment_keys.append('hidden_states')
            out['hidden_states'] = full_hidden_states

        return out

    def manage_gradients(self, memory_state, seg_num):
        k2, max_n_segments = self.rmt_config.get('k2'), self.rmt_config.get('max_n_segments')
        if seg_num == 0 \
            or k2 in {-1, None} \
                or seg_num + k2 > max_n_segments:
            return memory_state

        memory_state = memory_state.detach()
        return memory_state

    def gradient_checkpointing_enable(self, *args, **kwargs):
        self.memory_cell.model.gradient_checkpointing_enable(*args, **kwargs)


class StopOnSpecialTokenCriteria(StoppingCriteria):
    def __init__(self, special_token_ids):
        self.special_token_ids = set(special_token_ids)

    def __call__(self, input_ids, scores, **kwargs):
        last_token = input_ids[0, -1].item()
        return last_token in self.special_token_ids